# Accessing Rubin Data Preview 2 (DP2)

In this tutorial, we will:

- access Rubin's Data Preview 2 with LSDB
  * at RSP (Rubin Science Platform), based in the USA and the UK
  * at NERSC (National Energy Research Scientific Computing Center) for the [LSST DESC](https://lsstdesc.org) members

## Introduction

### Prerequisites

In order to access Rubin data, you must be a [Rubin data rights holder](https://rubinobservatory.org/for-scientists/data-products/data-policy).

## 1. Accessing the data on Rubin Science Platform (RSP)

### 1.1 Prepare your RSP container

- Visit https://data.lsst.cloud, unless you are accessing RSP through UK IDAC participation program - in that case, visit https://rsp.lsst.ac.uk.
- Log in using your identity provider.
- Once in, you will see Portal, Notebooks, and APIs.  Choose Notebooks.
- When it asks you what container to start, choose either "Recommended" or one of the latest weekly releases on the left, and "Large" on the right.
  - At the time of this writing, "Recommended" comes with an older version of lsdb - run the cell below to see if you'll need to target a weekly release instead.
- Once this has started, create a new notebook.

#### 1.1.1 Ensure your notebook kernel has the right version of lsdb

Make sure you've got at least **version 0.10.1 of lsdb**.  Try the following.

In [ ]:
%pip list | grep -E 'lsdb|hats'

In [ ]:
# Or for even more detail:

import lsdb

lsdb.show_versions()

<div class="alert alert-warning">
If the above shows a version that is very old, we suggest using RSP with a latest weekly release rather than the "Recommended" build.
</div>

### 1.2. Create a Dask Client

When working on RSP, with a Large container you have access to 32 GB of RAM. To ensure each worker has enough memory to work with the data, we recommend using 4 workers with 1 thread each, and memory limit of "auto" (which will divide the available memory across the workers). Dask also will sometimes spill to disk when it needs to, so we recommend setting the local directory to the `/deleted-sundays` directory, which is a large scratch space available on RSP that will be cleared every Sunday. You can set up your Dask client with the following code:

In [ ]:
# Dask puts out more advisory logging than we care for in this tutorial.
# It takes some doing to quiet all of it, but this recipe works.

import dask
import os

dask.config.set({"logging.distributed": "critical"})

import logging

# This also has to be done, for the above to be effective
logger = logging.getLogger("distributed")
logger.setLevel(logging.CRITICAL)

import warnings

# Finally, suppress the specific warning about Dask dashboard port usage
warnings.filterwarnings("ignore", message="Port 8787 is already in use.")

In [ ]:
from dask.distributed import Client

client = Client(
    n_workers=4,
    threads_per_worker=1,
    memory_limit="auto",
    local_directory=f"/deleted-sundays/{os.environ.get('USER', 'dask_scratch')}",
)
client

Your Dask dashboard will be accessible at `https://{username}.nb.data.lsst.cloud/nb/user/{username}/proxy/{port}/status`.

### 1.3 Opening a Catalog

The data is divided into `objects` and `dia_objects`.  Let's open both catalogs:

In [ ]:
from upath import UPath

base_path = UPath("/rubin/lsdb_data/dp2")

object_cat = lsdb.open_catalog(base_path / "object_collection")
dia_object_cat = lsdb.open_catalog(base_path / "dia_object_collection")

In [ ]:
object_cat

In [ ]:
dia_object_cat

You've accessed the data!  You can see the schema of both catalogs.  

To learn how to use this data,
see [Using Rubin Data](https://docs.lsdb.io/en/latest/tutorials/pre_executed/using_rubin_data.html) 
and follow the lsdb quickstart at [Getting Started](https://docs.lsdb.io/en/latest/getting-started.html). 


If you have questions about data from the Rubin Observatory, the best place to ask is https://community.lsst.org 
in the [LSST Data Products category](https://community.lsst.org/c/support/data/34).

## 2. Accessing the data at NERSC (Perlmutter)

If you are a part of the LSST DESC collaboration and have a NERSC account, you can access Rubin DP2 via Perlmutter cluster.
You can use both batch jobs and jupyter.nersc.gov, bellow we assume that you use NERSC's Jupyter Hub.

### 2.1 Launch Jupyter

Login to NERSC at https://jupyter.nersc.gov. Select "Login Node" for data exploration, configuration and code development. Use "Exclusive CPU Node" for larger tasks, such as full-catalog analysis.

Please also see NERSC documentation for [Dask configuration](https://gitlab.com/NERSC/nersc-notebooks/-/tree/main/perlmutter/dask).

### 2.2a Start kernel with LSDB

LSDB is available in a Jupyter kernel `desc-td_env-dev`.

If you haven't already set up the DESC Jupyter kernels at NERSC, run the one-time setup step on the Perlmutter command line:

```bash
source /global/common/software/lsst/common/miniconda/kernels/setup.sh
```

Then the next time you start up jupyter.nersc.gov, you'll have access to the `desc-td-env` jupyter kernel.



### 2.2b Alternative: Install LSDB

For conda installation run `conda install conda-forge::lsdb` in the terminal.
For pip installation run `python -m pip install lsdb` or the following cell in a Jupyter notebook:

In [ ]:
%pip install lsdb

Restart the kernel and check that the `lsdb` version is up-to-date:

In [ ]:
import lsdb

lsdb.__version__

### 2.4 Open a catalog

The data is divided into `objects` and `dia_objects`.  Let's open both catalogs:

In [6]:
from upath import UPath

base_path = UPath("/global/cfs/cdirs/lsst/shared/rubin/DP2/HATS/")

object_cat = lsdb.open_catalog(base_path / "object_collection")
dia_object_cat = lsdb.open_catalog(base_path / "dia_object_collection")
photoz_cat = lsdb.open_catalog(base_path / "object_photoz")

In [ ]:
object_cat

In [ ]:
dia_object_cat

In [ ]:
photoz_cat

## About

**Authors**: Neven Caplar, Sandro Campos, Olivia Lynn, Konstantin Malanchev

**Last updated on**: July 17, 2026

**Last run:** Never, waiting for DP2 release

If you use `lsdb` for published research, please cite following [instructions](https://docs.lsdb.io/en/stable/citation.html).